# **Week 1**

In [23]:
import zipfile
import os

# Specify the path to the zip file
zip_file_path = '/content/archive (3).zip'

# Specify the extraction directory
extract_dir = '/content/sales_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Open the zip file in read mode and extract all its contents
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Contents extracted to: {extract_dir}")


Contents extracted to: /content/sales_data


In [24]:
import os

extract_dir = '/content/sales_data'

# List the contents of the extracted directory
files_in_dir = os.listdir(extract_dir)

print(f"Files in '{extract_dir}':\n{files_in_dir}")


Files in '/content/sales_data':\n['9. Sales-Data-Analysis.csv']\n

# **Load Sales Data**

In [25]:
import pandas as pd
import os

# Construct the full path to the sales data CSV file
extract_dir = '/content/sales_data'
file_name = '9. Sales-Data-Analysis.csv'
file_path = os.path.join(extract_dir, file_name)

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
print("DataFrame Head:")
print(df.head())

# Print a concise summary of the DataFrame
print("\nDataFrame Info:")
df.info()


DataFrame Head:
   Order ID        Date             Product  Price  Quantity Purchase Type  \
0     10452  07-11-2022               Fries   3.49    573.07       Online    
1     10453  07-11-2022           Beverages   2.95    745.76       Online    
2     10454  07-11-2022       Sides & Other   4.99    200.40     In-store    
3     10455  08-11-2022             Burgers  12.99    569.67     In-store    
4     10456  08-11-2022  Chicken Sandwiches   9.95    201.01     In-store    

  Payment Method             Manager    City  
0      Gift Card    Tom      Jackson  London  
1      Gift Card         Pablo Perez  Madrid  
2      Gift Card       Joao    Silva  Lisbon  
3    Credit Card       Walter Muller  Berlin  
4    Credit Card       Walter Muller  Berlin  

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        2

# **Clean and Format Datetime Index**

In [26]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df = df.set_index('Date')
df = df.sort_index()

# Check for duplicate dates in the index
duplicate_dates = df.index.duplicated(keep=False)
if duplicate_dates.any():
    print("Duplicate dates found in the index. Here are the rows with duplicate dates:")
    print(df[duplicate_dates])
else:
    print("No duplicate dates found in the index.")

print("\nDataFrame after setting Date as index and sorting:")
print(df.head())
print("\nDataFrame Info after conversion and indexing:")
df.info()


Duplicate dates found in the index. Here are the rows with duplicate dates:
            Order ID             Product  Price  Quantity Purchase Type  \
Date                                                                      
2022-11-07     10452               Fries   3.49    573.07       Online    
2022-11-07     10453           Beverages   2.95    745.76       Online    
2022-11-07     10454       Sides & Other   4.99    200.40     In-store    
2022-11-08     10455             Burgers  12.99    569.67     In-store    
2022-11-08     10456  Chicken Sandwiches   9.95    201.01     In-store    
...              ...                 ...    ...       ...           ...   
2022-12-28     10706  Chicken Sandwiches   9.95    301.51   Drive-thru    
2022-12-29     10711  Chicken Sandwiches   9.95    281.41   Drive-thru    
2022-12-29     10712               Fries   3.49    630.37   Drive-thru    
2022-12-29     10710             Burgers  12.99    754.43   Drive-thru    
2022-12-29     10713    

In [27]:
# Replacement for cell 27 — compute per-transaction Sales, then aggregate by date correctly

# (Assumes `df` currently contains the transaction-level DataFrame with Date as DatetimeIndex)
# Make a copy to preserve transaction-level data for later per-product work
df_transactions = df.copy()

# 1) Compute Sales per transaction (correct revenue calculation)
df_transactions['Sales'] = df_transactions['Price'] * df_transactions['Quantity']

# 2) Aggregate transactions by date (sum Sales, sum Quantity, keep other columns as summary)
df_daily = df_transactions.groupby(df_transactions.index).agg({
    'Order ID': 'first',                       # keep one representative order id (or change to 'count' if preferred)
    'Product': lambda x: ', '.join(x.astype(str)),
    'Price': 'mean',                           # optional reference - do NOT use to compute Sales
    'Quantity': 'sum',                         # total quantity per day
    'Purchase Type': lambda x: ', '.join(x.astype(str)),
    'Payment Method': lambda x: ', '.join(x.astype(str)),
    'Manager': lambda x: ', '.join(x.astype(str)),
    'City': lambda x: ', '.join(x.astype(str)),
    'Sales': 'sum'                             # aggregate the per-row Sales (correct daily revenue)
})

# 3) Resample to daily frequency to ensure continuity and fill missing days for Sales with 0
df_daily = df_daily.resample('D').agg({
    'Sales': 'sum',
    'Order ID': lambda x: list(x.dropna().unique()),
    'Product': lambda x: ', '.join(x.dropna().unique()),
    'Price': 'mean',
    'Quantity': 'sum',
    'Purchase Type': lambda x: ', '.join(x.dropna().unique()),
    'Payment Method': lambda x: ', '.join(x.dropna().unique()),
    'Manager': lambda x: ', '.join(x.dropna().unique()),
    'City': lambda x: ', '.join(x.dropna().unique())
})

df_daily['Sales'] = df_daily['Sales'].fillna(0)

# 4) Verification: totals should match original transaction-level Sales sum
total_from_rows = df_transactions['Sales'].sum()
total_from_daily = df_daily['Sales'].sum()
print(f"Total revenue from transactions: {total_from_rows:.2f}")
print(f"Total revenue after daily aggregation: {total_from_daily:.2f}")

# 5) Check for missing dates (should be none after resample but we report if any)
full_date_range = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
missing_dates = full_date_range.difference(df_daily.index)

if not missing_dates.empty:
    print(f"Missing dates found in the index: {missing_dates}")
else:
    print("No missing dates found in the daily index — datetime index is continuous.")

# 6) Replace df with the daily DataFrame for subsequent cells (same variable name used later in notebook)
df = df_daily
print("
DataFrame after correct aggregation (daily):")
print(df.head())


No missing dates found in the index, the datetime index is continuous.

DataFrame after handling duplicate dates, calculating sales, and ensuring daily continuity:
                  Sales Order ID  \
Date                               
2022-11-07   5788.26630  [10452]   
2022-11-08  12129.29825  [10455]   
2022-11-09  15168.99328  [10460]   
2022-11-10  14736.42040  [10467]   
2022-11-11  15562.87348  [10470]   

                                                      Product  Price  \
Date                                                                   
2022-11-07                    Fries, Beverages, Sides & Other  3.810   
2022-11-08  Burgers, Chicken Sandwiches, Fries, Sides & Other  7.855   
2022-11-09  Burgers, Chicken Sandwiches, Fries, Beverages,...  6.874   
2022-11-10      Fries, Beverages, Burgers, Chicken Sandwiches  7.345   
2022-11-11  Burgers, Chicken Sandwiches, Fries, Beverages,...  6.874   


**Check for Missing Values and Outliers**

In [28]:
print("Missing values before handling (if any):")
print(df.isnull().sum())


Missing values before handling (if any):
Sales             0
Order ID          0
Product           0
Price             0
Quantity          0
Purchase Type     0
Payment Method    0
Manager           0
City              0
dtype: int64


# **Plot Overall Sales Trend**

In [29]:
import matplotlib.pyplot as plt
import seaborn as sns

# Generate a box plot of the 'Sales' column to visually identify potential outliers
plt.figure(figsize=(10, 6))
sns.boxplot(y=df['Sales'])
plt.title('Box Plot of Sales to Identify Outliers')
plt.ylabel('Sales')
plt.grid(True)
plt.show()


# **Decompose Time Series (Weekly Seasonality)**

In [30]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a line plot of the 'Sales' column over time
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Sales'], label='Daily Sales', color='blue')

# Add title and labels
plt.title('Overall Sales Trend Over Time')
plt.xlabel('Date')
plt.ylabel('Sales')

# Add legend and grid
plt.legend()
plt.grid(True)

# Display the plot
plt.show()


In [31]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Apply seasonal_decompose with an additive model and a weekly period (7 days)
decomposition = seasonal_decompose(df['Sales'], model='additive', period=7)

# Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Time Series Decomposition (Weekly Seasonality)', y=1.02)

# Add legends to each subplot if not automatically present
# The decomposition.plot() method usually adds titles and labels automatically,
# but we can explicitly set them or adjust if needed.

# Customize titles for better clarity
fig.axes[0].set_title('Original Sales')
fig.axes[1].set_title('Trend Component')
fig.axes[2].set_title('Seasonal Component (Weekly)')
fig.axes[3].set_title('Residual Component')

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()


In [32]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Apply seasonal_decompose with an additive model and an adjusted monthly period (26 days)
# Original request was for 30 days, but data length (53 observations) requires 2*period <= 53.
# 26 days is chosen as the largest approximate monthly period satisfying this (2*26 = 52).
decomposition = seasonal_decompose(df['Sales'], model='additive', period=26)

# Plot the decomposed components
fig = decomposition.plot()
fig.set_size_inches(12, 8)
fig.suptitle('Time Series Decomposition (Approx. Monthly Seasonality - 26 Days)', y=1.02)

# Customize titles for better clarity
fig.axes[0].set_title('Original Sales')
fig.axes[1].set_title('Trend Component')
fig.axes[2].set_title('Seasonal Component (Approx. Monthly)')
fig.axes[3].set_title('Residual Component')

plt.tight_layout(rect=[0, 0.03, 1, 0.98]) # Adjust layout to prevent suptitle overlap
plt.show()


# **Analyze Autocorrelation**

In [33]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Determine an appropriate number of lags based on the data length
# It's generally recommended to have lags less than N/2, where N is the number of observations.
# For safety, let's use min(len(df) - 1, 20) to ensure we don't exceed data length and keep plots readable.
lags_to_consider = min(len(df) - 1, 20)

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot ACF
plot_acf(df['Sales'], ax=axes[0], lags=lags_to_consider)
axes[0].set_title('Autocorrelation Function (ACF) of Sales')
axes[0].set_xlabel('Lags')
axes[0].set_ylabel('Autocorrelation')
axes[0].grid(True)

# Plot PACF
plot_pacf(df['Sales'], ax=axes[1], lags=lags_to_consider)
axes[1].set_title('Partial Autocorrelation Function (PACF) of Sales')
axes[1].set_xlabel('Lags')
axes[1].set_ylabel('Partial Autocorrelation')
axes[1].grid(True)

plt.tight_layout()
plt.show()
